In [0]:
import matplotlib.pyplot as plt
import os
 

In [0]:
CATALOG = "venturescope-warehouse"   # <-- change to your actual catalog name
SCHEMA = "venturescope_marts"
 
if CATALOG:
    spark.sql(f"USE CATALOG `{CATALOG}`")
    TABLE_PREFIX = f"`{CATALOG}`.{SCHEMA}"
else:
    TABLE_PREFIX = SCHEMA
 
print(f"Using table prefix: {TABLE_PREFIX}")

In [0]:
# Query 1 — The Headline: What Actually Happens to Startups?
# ------------------------------------------------------------
q1 = f"""
SELECT
  outcome,
  COUNT(*) AS startups,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_of_total,
  ROUND(AVG(total_funding_usd) / 1e6, 1) AS avg_funding_m,
  ROUND(AVG(funding_rounds), 1) AS avg_rounds
FROM {TABLE_PREFIX}.startup_outcomes
GROUP BY outcome
ORDER BY startups DESC
"""
df_q1 = spark.sql(q1)
df_q1.display()
pdf_q1 = df_q1.toPandas()
 
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(pdf_q1["outcome"], pdf_q1["startups"], color="#4C72B0")
ax.set_title("What Actually Happens to Startups?")
ax.set_xlabel("Outcome")
ax.set_ylabel("Number of Startups")
plt.xticks(rotation=30, ha="right")
for bar, pct in zip(bars, pdf_q1["pct_of_total"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
             f"{pct}%", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

In [0]:
# Query 2 — Does More Money Mean Better Odds?
# ------------------------------------------------------------
q2 = f"""
SELECT
  funding_bucket,
  COUNT(*) AS startups,
  ROUND(AVG(is_successful_exit) * 100, 1) AS success_rate_pct,
  ROUND(AVG(is_closed) * 100, 1) AS closure_rate_pct,
  ROUND(AVG(is_ipo) * 100, 1) AS ipo_rate_pct
FROM {TABLE_PREFIX}.startup_outcomes
WHERE funding_bucket != 'No Data'
GROUP BY funding_bucket
ORDER BY
  CASE funding_bucket
    WHEN 'Bootstrapped' THEN 1
    WHEN 'Under $1M' THEN 2
    WHEN '$1M–$10M' THEN 3
    WHEN '$10M–$50M' THEN 4
    WHEN '$50M–$200M' THEN 5
    WHEN '$200M+' THEN 6
  END
"""
df_q2 = spark.sql(q2)
df_q2.display()
pdf_q2 = df_q2.toPandas()
 
fig, ax = plt.subplots(figsize=(9, 5))
x = range(len(pdf_q2))
width = 0.25
ax.bar([i - width for i in x], pdf_q2["success_rate_pct"], width, label="Success Rate %", color="#55A868")
ax.bar(x, pdf_q2["closure_rate_pct"], width, label="Closure Rate %", color="#C44E52")
ax.bar([i + width for i in x], pdf_q2["ipo_rate_pct"], width, label="IPO Rate %", color="#4C72B0")
ax.set_xticks(list(x))
ax.set_xticklabels(pdf_q2["funding_bucket"], rotation=30, ha="right")
ax.set_ylabel("Percent")
ax.set_title("Does More Money Mean Better Odds?")
ax.legend()
plt.tight_layout()
plt.show()

In [0]:
# Query 3 — Which Industries Survive Best?
# ------------------------------------------------------------
q3 = f"""
SELECT
  primary_category,
  total_startups,
  success_rate_pct,
  closure_rate_pct,
  ipo_rate_pct,
  avg_funding_m
FROM {TABLE_PREFIX}.category_survival_rates
ORDER BY success_rate_pct DESC
LIMIT 15
"""
df_q3 = spark.sql(q3)
df_q3.display()
pdf_q3 = df_q3.toPandas()
 
fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(pdf_q3["primary_category"][::-1], pdf_q3["success_rate_pct"][::-1], color="#55A868")
ax.set_xlabel("Success Rate %")
ax.set_title("Top 15 Industries by Startup Success Rate")
plt.tight_layout()
plt.show()

In [0]:
# Query 4 — Window Function: Cumulative Funding by Founding Year
# ------------------------------------------------------------
q4 = f"""
WITH yearly AS (
  SELECT
    founded_year,
    COUNT(*) AS startups_founded,
    SUM(total_funding_usd) / 1e9 AS total_funding_bn,
    AVG(is_successful_exit) * 100 AS success_rate
  FROM {TABLE_PREFIX}.startup_outcomes
  WHERE founded_year BETWEEN 1990 AND 2015
  GROUP BY founded_year
)
SELECT
  founded_year,
  startups_founded,
  ROUND(total_funding_bn, 1) AS total_funding_bn,
  ROUND(success_rate, 1) AS success_rate_pct,
  ROUND(SUM(total_funding_bn) OVER (
    ORDER BY founded_year
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  ), 1) AS cumulative_funding_bn
FROM yearly
ORDER BY founded_year
"""
df_q4 = spark.sql(q4)
df_q4.display()
pdf_q4 = df_q4.toPandas()
 
fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.bar(pdf_q4["founded_year"], pdf_q4["startups_founded"], color="#DD8452", alpha=0.6, label="Startups Founded")
ax1.set_xlabel("Founded Year")
ax1.set_ylabel("Startups Founded", color="#DD8452")
ax1.tick_params(axis="y", labelcolor="#DD8452")
 
ax2 = ax1.twinx()
ax2.plot(pdf_q4["founded_year"], pdf_q4["cumulative_funding_bn"], color="#4C72B0", linewidth=2.5, label="Cumulative Funding ($B)")
ax2.set_ylabel("Cumulative Funding ($B)", color="#4C72B0")
ax2.tick_params(axis="y", labelcolor="#4C72B0")
 
ax1.set_title("Cumulative Funding by Founding Year (1990–2015)")
plt.tight_layout()
plt.show()

In [0]:
# Query 5 — The Speed Question: Does Getting Funded Fast Matter?
# ------------------------------------------------------------
q5 = f"""
SELECT
  speed_to_funding_bucket,
  COUNT(*) AS startups,
  ROUND(AVG(is_successful_exit) * 100, 1) AS success_rate_pct,
  ROUND(AVG(is_closed) * 100, 1) AS closure_rate_pct,
  ROUND(AVG(total_funding_usd) / 1e6, 1) AS avg_funding_m
FROM {TABLE_PREFIX}.startup_outcomes
WHERE speed_to_funding_bucket != 'Unknown'
  AND speed_to_funding_bucket != 'Pre-founded'
GROUP BY speed_to_funding_bucket
ORDER BY
  CASE speed_to_funding_bucket
    WHEN 'Under 6 months' THEN 1
    WHEN '6–12 months' THEN 2
    WHEN '1–2 years' THEN 3
    WHEN '2–5 years' THEN 4
    WHEN '5+ years' THEN 5
  END
"""
df_q5 = spark.sql(q5)
df_q5.display()
pdf_q5 = df_q5.toPandas()
 
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(pdf_q5["speed_to_funding_bucket"], pdf_q5["success_rate_pct"], marker="o", color="#55A868", label="Success Rate %")
ax.plot(pdf_q5["speed_to_funding_bucket"], pdf_q5["closure_rate_pct"], marker="o", color="#C44E52", label="Closure Rate %")
ax.set_xlabel("Speed to Funding")
ax.set_ylabel("Percent")
ax.set_title("Does Getting Funded Fast Matter?")
ax.legend()
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

In [0]:
# Query 6 — Geography: US vs The World
# ------------------------------------------------------------
q6 = f"""
SELECT
  region,
  COUNT(*) AS startups,
  ROUND(AVG(is_successful_exit) * 100, 1) AS success_rate_pct,
  ROUND(AVG(is_ipo) * 100, 1) AS ipo_rate_pct,
  ROUND(AVG(total_funding_usd) / 1e6, 1) AS avg_funding_m,
  ROUND(AVG(funding_rounds), 1) AS avg_rounds
FROM {TABLE_PREFIX}.startup_outcomes
GROUP BY region
HAVING COUNT(*) >= 50
ORDER BY success_rate_pct DESC
"""
df_q6 = spark.sql(q6)
df_q6.display()
pdf_q6 = df_q6.toPandas()
 
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(pdf_q6["region"][::-1], pdf_q6["success_rate_pct"][::-1], color="#4C72B0")
ax.set_xlabel("Success Rate %")
ax.set_title("Startup Success Rate by Region")
plt.tight_layout()
plt.show()

In [0]:
# Query 7 — The Contrarian: Efficient Raisers vs Big Raisers
# ------------------------------------------------------------
q7 = f"""
SELECT
  quartile_label,
  startups,
  avg_funding_m,
  avg_rounds,
  success_rate_pct,
  ipo_rate_pct,
  closure_rate_pct
FROM {TABLE_PREFIX}.funding_efficiency
ORDER BY efficiency_quartile
"""
df_q7 = spark.sql(q7)
df_q7.display()
pdf_q7 = df_q7.toPandas()
 
fig, ax = plt.subplots(figsize=(9, 5))
x = range(len(pdf_q7))
width = 0.25
ax.bar([i - width for i in x], pdf_q7["success_rate_pct"], width, label="Success Rate %", color="#55A868")
ax.bar(x, pdf_q7["ipo_rate_pct"], width, label="IPO Rate %", color="#4C72B0")
ax.bar([i + width for i in x], pdf_q7["closure_rate_pct"], width, label="Closure Rate %", color="#C44E52")
ax.set_xticks(list(x))
ax.set_xticklabels(pdf_q7["quartile_label"], rotation=20, ha="right")
ax.set_ylabel("Percent")
ax.set_title("Efficient Raisers vs Big Raisers")
ax.legend()
plt.tight_layout()
plt.show()